# Approach 3 — Font-to-Handwriting CNN Training

**Before running:** Runtime → Change runtime type → **T4 GPU**

### What this trains
`HandwritingStyleNet` (U-Net encoder-decoder):  
Input: clean font-rendered character image (PIL/FreeType)  
Output: handwriting-style image matching the target writer  
Loss: L1 (pixel-level MAE). Writer style injected via embedding at bottleneck.

### After training
Download `style_net.pt` and place it at:  
`approaches/03_FontToHandwriting_CNN/checkpoints/style_net.pt`  
Then run locally: `cd approaches/03_FontToHandwriting_CNN && python run.py`

In [ ]:
# Cell 1: Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Install dependencies
!pip install scipy scikit-image -q
print('Dependencies installed')

In [ ]:
# Cell 4: Clone repo
import os

REPO      = 'AI-Powered-Handwriting-Generation-System'
REPO_PATH = f'/content/{REPO}'

if os.path.exists(REPO_PATH):
    print('Repo exists — pulling latest...')
    os.system(f'git -C {REPO_PATH} pull')
else:
    print('Cloning repo...')
    os.system(f'git clone --depth 1 https://github.com/Abdullahshaz70/{REPO}.git {REPO_PATH}')

APPROACH_DIR = f'{REPO_PATH}/approaches/03_FontToHandwriting_CNN'
print('Approach dir:', APPROACH_DIR)

In [ ]:
# Cell 5: Configure training
EPOCHS = 80   # 60-100 recommended

print(f'Will train for {EPOCHS} epochs')
print(f'Data: {REPO_PATH}/Data/Writers_pngs')

In [ ]:
# Cell 6: Train
os.chdir(APPROACH_DIR)
!python run.py --train --epochs {EPOCHS}

In [ ]:
# Cell 7: Save checkpoint to Drive + download
import shutil
from google.colab import files

ckpt_src  = f'{APPROACH_DIR}/checkpoints/style_net.pt'
drive_dir = '/content/drive/MyDrive/handwriting_checkpoints/03_FontCNN'
os.makedirs(drive_dir, exist_ok=True)

shutil.copy(ckpt_src, os.path.join(drive_dir, 'style_net.pt'))
print('Saved to Drive:', drive_dir)

files.download(ckpt_src)
print('\nDownload started.')
print('Place style_net.pt in:  approaches/03_FontToHandwriting_CNN/checkpoints/')
print('Then run locally:       cd approaches/03_FontToHandwriting_CNN && python run.py')

In [ ]:
# Cell 8: Preview — generate a-j for each writer
import sys
sys.path.insert(0, f'{APPROACH_DIR}/src')

import torch
import matplotlib.pyplot as plt
import numpy as np

from model    import HandwritingStyleNet
from generate import generate_char

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt   = torch.load(f'{APPROACH_DIR}/checkpoints/style_net.pt', map_location=device)
model  = HandwritingStyleNet(num_writers=ckpt['num_writers']).to(device)
model.load_state_dict(ckpt['model_state']); model.eval()

writer_names = ckpt['writer_names']
test_chars   = list('abcdefghij')

fig, axes = plt.subplots(len(writer_names), len(test_chars),
                          figsize=(len(test_chars)*1.5, len(writer_names)*1.5))
if len(writer_names) == 1:
    axes = [axes]

for wi, wname in enumerate(writer_names):
    for ci, ch in enumerate(test_chars):
        arr = generate_char(model, ch, wi, device)
        axes[wi][ci].imshow(arr, cmap='gray', vmin=0, vmax=255)
        if wi == 0: axes[wi][ci].set_title(ch)
        if ci == 0: axes[wi][ci].set_ylabel(wname.replace('writer_',''), fontsize=7)
        axes[wi][ci].axis('off')

plt.suptitle(f'Font-to-Handwriting CNN — {EPOCHS} epochs')
plt.tight_layout()
plt.show()